# 🧠 MetaBridge — AI Data Catalog Generator
### Step-by-step demo using Northwind SQLite (zero setup required)

**What this notebook does:**
1. Downloads the free Northwind SQLite database automatically
2. Profiles every table (schema, row counts, sample data, nulls, cardinality)
3. Sends each table profile to an LLM to generate business descriptions
4. Displays enriched metadata with PII flags, domain, sensitivity
5. Writes catalog JSON files + lineage log locally
6. (Optional) Pushes to Microsoft Purview or Databricks Unity Catalog

> ⚡ **Run without an LLM key first** — mock mode will generate placeholder descriptions so you can see the full pipeline working.
> Add your key in Cell 3 when ready for real AI descriptions.

---
## 📦 Step 1: Install Dependencies

In [ ]:
import subprocess, sys

pkgs = ["openai", "sqlalchemy", "pandas", "requests", "rich", "tabulate"]
for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ All dependencies installed successfully")

---
## ⚙️ Step 2: Imports & Config

In [ ]:
import os, json, uuid, time, urllib.request, warnings
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Optional

import pandas as pd
import sqlalchemy as sa
from sqlalchemy import inspect, text
from IPython.display import display, HTML, Markdown

warnings.filterwarnings("ignore")
print("✅ Imports done")

---
## 🔑 Step 3: Configuration

> **Add your LLM API key below when ready.**  
> Leave as `ADD_YOUR_KEY_HERE` to run in **mock mode** (no API calls, placeholder descriptions).

In [ ]:
CONFIG = {
    # ── LLM Settings ─────────────────────────────────────────────────
    # Provider options: "openai" | "azure_openai" | "ollama"
    "llm_provider"            : "openai",
    "llm_api_key"             : "ADD_YOUR_KEY_HERE",   # ← paste your key here
    "llm_model"               : "gpt-4o-mini",         # cheap & fast

    # Azure OpenAI (only if using azure_openai above)
    "azure_openai_endpoint"   : "",
    "azure_openai_api_version": "2024-02-01",

    # Ollama (local, free, no key needed — change provider to 'ollama')
    "ollama_model"            : "llama3",
    "ollama_base_url"         : "http://localhost:11434/v1",

    # ── Source Database ───────────────────────────────────────────────
    "source_name"             : "Northwind_SQLite",
    "db_path"                 : "./northwind.db",      # auto-downloaded
    "sample_rows"             : 15,                    # rows sampled per table
    "max_tables"              : None,                  # set to 3 to test faster

    # ── Output ────────────────────────────────────────────────────────
    "output_dir"              : "./metabridge_output",

    # ── Catalog Targets ───────────────────────────────────────────────
    # "file" = write JSON locally (default, no credentials needed)
    # Add "purview" or "unity_catalog" + fill credentials below
    "targets"                 : ["file"],

    "purview_account"         : "",   # e.g. "my-purview-account"
    "databricks_host"         : "",   # e.g. "https://adb-xxx.azuredatabricks.net"
    "databricks_token"        : "",
    "databricks_warehouse_id" : "",
    "uc_catalog"              : "northwind",
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)

mock_mode = CONFIG["llm_api_key"] == "ADD_YOUR_KEY_HERE" and CONFIG["llm_provider"] != "ollama"
display(HTML(f"""
<div style='background:#1a1a2e;padding:15px;border-radius:8px;color:white;font-family:monospace'>
  <b style='font-size:16px'>⚙️ MetaBridge Configuration</b><br><br>
  🤖 LLM Provider : <span style='color:#00d4ff'>{CONFIG['llm_provider']}</span><br>
  🔑 Mode         : <span style='color:{"#ff6b6b" if mock_mode else "#51cf66"}'>{'🟡 MOCK (no key)' if mock_mode else '🟢 LIVE (key set)'}</span><br>
  📦 Source       : <span style='color:#ffd43b'>{CONFIG['source_name']}</span><br>
  💾 Output       : <span style='color:#a9e34b'>{CONFIG['output_dir']}</span><br>
  🎯 Targets      : <span style='color:#74c0fc'>{', '.join(CONFIG['targets'])}</span>
</div>
"""))

---
## 🗄️ Step 4: Download Northwind SQLite Database

Northwind is Microsoft's classic sample database (customers, orders, products, suppliers).  
We use the SQLite version from [jpwhite3/northwind-SQLite3](https://github.com/jpwhite3/northwind-SQLite3) — **zero setup, no server needed.**

In [ ]:
DB_URL = "https://github.com/jpwhite3/northwind-SQLite3/raw/main/dist/northwind.db"

if os.path.exists(CONFIG["db_path"]):
    size_kb = os.path.getsize(CONFIG["db_path"]) // 1024
    display(HTML(f"<div style='color:green;font-weight:bold'>✅ Northwind DB already exists ({size_kb} KB) — skipping download</div>"))
else:
    display(HTML("<div style='color:orange'>⬇️ Downloading Northwind SQLite from GitHub...</div>"))
    start = time.time()
    urllib.request.urlretrieve(DB_URL, CONFIG["db_path"])
    elapsed = round(time.time() - start, 1)
    size_kb = os.path.getsize(CONFIG["db_path"]) // 1024
    display(HTML(f"<div style='color:green;font-weight:bold'>✅ Downloaded northwind.db ({size_kb} KB) in {elapsed}s</div>"))

# Quick connection test
engine = sa.create_engine(f"sqlite:///{CONFIG['db_path']}")
with engine.connect() as conn:
    result = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table'")).fetchall()
    table_names = [r[0] for r in result]

display(HTML(f"""
<div style='background:#0d1117;padding:12px;border-left:4px solid #238636;border-radius:4px;color:white;margin-top:10px'>
  <b>📊 Northwind Database Connected</b><br>
  Tables found: <span style='color:#58a6ff'>{len(table_names)}</span><br>
  <span style='color:#8b949e'>{', '.join(table_names)}</span>
</div>
"""))

---
## 🔬 Step 5: Define Data Models

In [ ]:
@dataclass
class ColumnProfile:
    name: str
    data_type: str
    nullable: bool
    sample_values: List[str] = field(default_factory=list)
    null_percentage: float = 0.0
    cardinality: int = 0
    ai_description: str = ""
    is_pii: bool = False

@dataclass
class TableProfile:
    source_system: str
    database: str
    schema: str
    table_name: str
    row_count: int
    columns: List[ColumnProfile] = field(default_factory=list)
    primary_keys: List[str] = field(default_factory=list)
    foreign_keys: List[dict] = field(default_factory=list)
    ai_table_description: str = ""
    ai_table_purpose: str = ""
    ai_domain: str = ""
    ai_sensitivity: str = ""
    ai_tags: List[str] = field(default_factory=list)
    lineage_run_id: str = ""

print("✅ Data models defined: ColumnProfile, TableProfile")

---
## 🔍 Step 6: Table Profiler

Reads schema, row counts, sample data, null rates, and cardinality from each source table.

In [ ]:
class NorthwindProfiler:
    def __init__(self, db_path, source_name, sample_rows=15):
        self.engine = sa.create_engine(f"sqlite:///{db_path}")
        self.source_name = source_name
        self.sample_rows = sample_rows

    def get_tables(self):
        return inspect(self.engine).get_table_names()

    def profile_table(self, table_name):
        insp = inspect(self.engine)
        cols_meta = insp.get_columns(table_name)
        try:
            pk = insp.get_pk_constraint(table_name)
            primary_keys = pk.get("constrained_columns", [])
        except Exception:
            primary_keys = []
        try:
            fks = insp.get_foreign_keys(table_name)
        except Exception:
            fks = []

        with self.engine.connect() as conn:
            row_count = conn.execute(text(f'SELECT COUNT(*) FROM "{table_name}"')).scalar() or 0
            try:
                sample_df = pd.read_sql(f'SELECT * FROM "{table_name}" LIMIT {self.sample_rows}', conn)
            except Exception:
                sample_df = pd.DataFrame()

        columns = []
        for col in cols_meta:
            cname = col["name"]
            if cname in sample_df.columns:
                series = sample_df[cname]
                null_pct  = round(series.isna().mean() * 100, 1)
                cardinality = series.nunique()
                samples = series.dropna().astype(str).unique()[:5].tolist()
            else:
                null_pct, cardinality, samples = 0.0, 0, []

            columns.append(ColumnProfile(
                name=cname,
                data_type=str(col.get("type", "UNKNOWN")),
                nullable=col.get("nullable", True),
                sample_values=samples,
                null_percentage=null_pct,
                cardinality=cardinality
            ))

        return TableProfile(
            source_system=self.source_name,
            database="northwind",
            schema="main",
            table_name=table_name,
            row_count=row_count,
            columns=columns,
            primary_keys=primary_keys,
            foreign_keys=fks,
            lineage_run_id=str(uuid.uuid4())
        )

profiler = NorthwindProfiler(CONFIG["db_path"], CONFIG["source_name"], CONFIG["sample_rows"])
all_tables = profiler.get_tables()
if CONFIG["max_tables"]:
    all_tables = all_tables[:CONFIG["max_tables"]]

print(f"✅ Profiler ready | {len(all_tables)} tables queued: {all_tables}")

---
## 🔭 Step 7: Profile a Single Table (Interactive Preview)

Let's deeply inspect the **Customers** table before running the full pipeline.

In [ ]:
# Preview a single table profile
preview_table = "Customers"  # change to any table name
profile = profiler.profile_table(preview_table)

display(HTML(f"""
<div style='background:#0d1117;padding:15px;border-radius:8px;color:white;font-family:monospace'>
  <span style='font-size:18px;font-weight:bold'>📋 {profile.table_name}</span><br>
  <span style='color:#8b949e'>Source: {profile.source_system} | Schema: {profile.schema} | DB: {profile.database}</span><br><br>
  <span style='color:#58a6ff'>Rows     :</span> {profile.row_count:,}<br>
  <span style='color:#58a6ff'>Columns  :</span> {len(profile.columns)}<br>
  <span style='color:#58a6ff'>PK       :</span> {profile.primary_keys}<br>
  <span style='color:#58a6ff'>FK count :</span> {len(profile.foreign_keys)}<br>
  <span style='color:#58a6ff'>Run ID   :</span> {profile.lineage_run_id}
</div>
"""))

# Show column profiles as a DataFrame
col_df = pd.DataFrame([{
    "Column"        : c.name,
    "Type"          : c.data_type,
    "Nullable"      : "✅" if c.nullable else "❌",
    "Null %"        : f"{c.null_percentage}%",
    "Cardinality"   : c.cardinality,
    "Sample Values" : " | ".join(c.sample_values[:3])
} for c in profile.columns])

display(HTML("<h4 style='margin-top:15px'>📊 Column Profiles</h4>"))
display(col_df.style.set_properties(**{
    'background-color': '#161b22',
    'color': 'white',
    'border': '1px solid #30363d'
}).set_table_styles([{
    'selector': 'th',
    'props': [('background-color', '#21262d'), ('color', '#58a6ff'), ('font-weight', 'bold')]
}]))

---
## 🤖 Step 8: AI Description Generator

Sends table profile to LLM → returns structured JSON with business descriptions, domain, sensitivity, PII flags.

In [ ]:
class AIDescriptionGenerator:
    def __init__(self, config):
        self.config   = config
        self.provider = config["llm_provider"]
        self._client  = None
        self._model   = config["llm_model"]
        self._init_client()

    def _init_client(self):
        if self.provider == "openai":
            from openai import OpenAI
            self._client = OpenAI(api_key=self.config["llm_api_key"])
        elif self.provider == "azure_openai":
            from openai import AzureOpenAI
            self._client = AzureOpenAI(
                azure_endpoint=self.config["azure_openai_endpoint"],
                api_key=self.config["llm_api_key"],
                api_version=self.config["azure_openai_api_version"]
            )
        elif self.provider == "ollama":
            from openai import OpenAI
            self._client = OpenAI(
                base_url=self.config.get("ollama_base_url", "http://localhost:11434/v1"),
                api_key="ollama"
            )
            self._model = self.config.get("ollama_model", "llama3")

    def _build_prompt(self, profile):
        col_lines = "\n".join([
            f"  - {c.name} ({c.data_type}) | nulls:{c.null_percentage}% | cardinality:{c.cardinality} | samples:{c.sample_values}"
            for c in profile.columns
        ])
        fk_text = json.dumps(profile.foreign_keys, indent=2) if profile.foreign_keys else "None"
        return f"""You are a senior data architect writing metadata for an enterprise data catalog.
Analyze this database table and generate rich business-friendly descriptions.

SOURCE: {profile.source_system} | TABLE: {profile.table_name} | ROWS: {profile.row_count:,}
PRIMARY KEYS: {profile.primary_keys}
FOREIGN KEYS: {fk_text}
COLUMNS:
{col_lines}

Return ONLY valid JSON with these exact keys:
{{
  "table_description": "<2-3 sentence business description>",
  "table_purpose": "<one-line purpose>",
  "domain": "<Finance|HR|Sales|Inventory|Logistics|Product|Customer|Other>",
  "sensitivity": "<Public|Internal|Confidential|Restricted>",
  "tags": ["tag1", "tag2", "tag3"],
  "pii_columns": ["col1", ...],
  "column_descriptions": {{ "col_name": "description", ... }}
}}
column_descriptions MUST include ALL {len(profile.columns)} columns. Return ONLY JSON."""

    def generate(self, profile):
        is_mock = (
            self.config["llm_api_key"] == "ADD_YOUR_KEY_HERE"
            and self.provider != "ollama"
        )
        if is_mock:
            return self._mock_generate(profile)
        try:
            resp = self._client.chat.completions.create(
                model=self._model,
                messages=[{"role": "user", "content": self._build_prompt(profile)}],
                temperature=0.2,
                response_format={"type": "json_object"}
            )
            return json.loads(resp.choices[0].message.content)
        except Exception as e:
            print(f"  ⚠️  LLM failed for {profile.table_name}: {e} → using mock")
            return self._mock_generate(profile)

    def _mock_generate(self, profile):
        pii_keywords = ["name", "email", "phone", "address", "contact", "city", "postal", "fax"]
        pii_cols = [c.name for c in profile.columns
                    if any(kw in c.name.lower() for kw in pii_keywords)]
        return {
            "table_description": (
                f"[MOCK] The {profile.table_name} table contains {profile.row_count:,} rows "
                f"sourced from {profile.source_system}. Add your LLM API key in CONFIG to "
                f"generate real AI descriptions for this table."
            ),
            "table_purpose": f"[MOCK] Stores {profile.table_name.lower()} records",
            "domain": "Other",
            "sensitivity": "Internal",
            "tags": [profile.table_name, "northwind", "source-system"],
            "pii_columns": pii_cols,
            "column_descriptions": {
                c.name: f"[MOCK] {c.name} ({c.data_type}) — samples: {c.sample_values[:2]}"
                for c in profile.columns
            }
        }

ai_gen = AIDescriptionGenerator(CONFIG)
mode   = "🟡 MOCK" if CONFIG["llm_api_key"] == "ADD_YOUR_KEY_HERE" else "🟢 LIVE"
print(f"✅ AI Generator ready | Mode: {mode} | Provider: {CONFIG['llm_provider']} | Model: {CONFIG['llm_model']}")

---
## 🧪 Step 9: Test AI on a Single Table (Customers)

In [ ]:
print(f"🤖 Generating description for: {profile.table_name}...")
t0 = time.time()
ai_meta = ai_gen.generate(profile)
elapsed = round(time.time() - t0, 2)
print(f"⏱️  Done in {elapsed}s")

# Show result
display(HTML(f"""
<div style='background:#0d1117;padding:15px;border-radius:8px;color:white;font-family:sans-serif;margin-top:10px'>
  <div style='font-size:17px;font-weight:bold;color:#58a6ff'>🏷️ {profile.table_name} — AI Generated Metadata</div>
  <hr style='border-color:#30363d'>
  <b>📝 Description:</b><br>
  <span style='color:#e6edf3'>{ai_meta.get('table_description','')}</span><br><br>
  <b>🎯 Purpose:</b> <span style='color:#a5d6ff'>{ai_meta.get('table_purpose','')}</span><br>
  <b>🏢 Domain:</b> <span style='color:#ffd700'>{ai_meta.get('domain','')}</span> &nbsp;
  <b>🔐 Sensitivity:</b> <span style='color:#ff7b72'>{ai_meta.get('sensitivity','')}</span><br>
  <b>🏷️ Tags:</b> <span style='color:#7ee787'>{', '.join(ai_meta.get('tags', []))}</span><br>
  <b>⚠️ PII Columns:</b> <span style='color:#ff6b6b'>{', '.join(ai_meta.get('pii_columns', [])) or 'None detected'}</span>
</div>
"""))

# Show column-level descriptions
col_descs = ai_meta.get("column_descriptions", {})
pii_set   = set(c.lower() for c in ai_meta.get("pii_columns", []))

desc_df = pd.DataFrame([{
    "Column"      : c.name,
    "Type"        : c.data_type,
    "PII"         : "🔴" if c.name.lower() in pii_set else "✅",
    "AI Description": col_descs.get(c.name, "—")
} for c in profile.columns])

display(HTML("<h4>📋 Column-Level AI Descriptions</h4>"))
display(desc_df)

---
## 🗂️ Step 10: Catalog Writers

In [ ]:
def enrich_profile(profile, ai_meta):
    profile.ai_table_description = ai_meta.get("table_description", "")
    profile.ai_table_purpose     = ai_meta.get("table_purpose", "")
    profile.ai_domain            = ai_meta.get("domain", "")
    profile.ai_sensitivity       = ai_meta.get("sensitivity", "")
    profile.ai_tags              = ai_meta.get("tags", [])
    pii_set  = {c.lower() for c in ai_meta.get("pii_columns", [])}
    col_desc = ai_meta.get("column_descriptions", {})
    for col in profile.columns:
        col.ai_description = col_desc.get(col.name, "")
        col.is_pii         = col.name.lower() in pii_set
    return profile


class FileCatalogWriter:
    def __init__(self, output_dir):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def write(self, profile):
        payload = {
            "metabridge_version": "1.0",
            "generated_at": datetime.utcnow().isoformat() + "Z",
            "source": {
                "system": profile.source_system,
                "database": profile.database,
                "schema": profile.schema,
                "table": profile.table_name,
                "row_count": profile.row_count,
                "primary_keys": profile.primary_keys,
                "foreign_keys": profile.foreign_keys,
            },
            "ai_metadata": {
                "description" : profile.ai_table_description,
                "purpose"     : profile.ai_table_purpose,
                "domain"      : profile.ai_domain,
                "sensitivity" : profile.ai_sensitivity,
                "tags"        : profile.ai_tags,
            },
            "columns": [{
                "name"          : c.name,
                "data_type"     : c.data_type,
                "nullable"      : c.nullable,
                "null_pct"      : c.null_percentage,
                "cardinality"   : c.cardinality,
                "sample_values" : c.sample_values,
                "ai_description": c.ai_description,
                "is_pii"        : c.is_pii,
            } for c in profile.columns],
            "lineage": {
                "run_id"            : profile.lineage_run_id,
                "source_to_catalog" : f"{profile.source_system}/{profile.schema}/{profile.table_name} → catalog",
                "processed_at"      : datetime.utcnow().isoformat() + "Z",
            }
        }
        path = f"{self.output_dir}/{profile.table_name}_catalog.json"
        with open(path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)
        return path


class LineageStore:
    def __init__(self, output_dir):
        self.log_path     = f"{output_dir}/lineage_log.jsonl"
        self.summary_path = f"{output_dir}/run_summary.json"
        self.records      = []

    def record(self, profile, status, targets, error=""):
        entry = {
            "run_id"        : profile.lineage_run_id,
            "timestamp"     : datetime.utcnow().isoformat() + "Z",
            "source_system" : profile.source_system,
            "source_table"  : f"{profile.schema}.{profile.table_name}",
            "row_count"     : profile.row_count,
            "columns_count" : len(profile.columns),
            "pii_columns"   : [c.name for c in profile.columns if c.is_pii],
            "ai_domain"     : profile.ai_domain,
            "ai_sensitivity": profile.ai_sensitivity,
            "targets"       : targets,
            "status"        : status,
            "error"         : error,
        }
        self.records.append(entry)
        with open(self.log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry) + "\n")
        return entry

    def write_summary(self, total, success, failed, duration):
        summary = {
            "run_date"      : datetime.utcnow().isoformat() + "Z",
            "total_tables"  : total,
            "success"       : success,
            "failed"        : failed,
            "duration_secs" : round(duration, 2),
            "tables"        : self.records
        }
        with open(self.summary_path, "w") as f:
            json.dump(summary, f, indent=2)
        return self.summary_path


file_writer = FileCatalogWriter(CONFIG["output_dir"])
lineage     = LineageStore(CONFIG["output_dir"])
print("✅ Catalog writers and lineage store initialized")

---
## 🚀 Step 11: Run Full Pipeline — All Tables

This processes every table in Northwind end-to-end:  
`Profile → AI Describe → Enrich → Write Catalog → Record Lineage`

In [ ]:
from IPython.display import clear_output

pipeline_start = time.time()
results        = []
success_count  = 0
failed_count   = 0

for i, table_name in enumerate(all_tables, 1):
    table_start = time.time()
    print(f"\n[{i}/{len(all_tables)}] ⏳ Processing: {table_name}")

    try:
        # 1. Profile
        tbl_profile = profiler.profile_table(table_name)
        print(f"  ✅ Profiled   → {tbl_profile.row_count:,} rows | {len(tbl_profile.columns)} cols | PK: {tbl_profile.primary_keys}")

        # 2. AI Generate
        tbl_ai_meta = ai_gen.generate(tbl_profile)
        print(f"  🤖 AI Done    → Domain: {tbl_ai_meta.get('domain')} | Sensitivity: {tbl_ai_meta.get('sensitivity')} | PII cols: {len(tbl_ai_meta.get('pii_columns',[]))}")

        # 3. Enrich
        tbl_profile = enrich_profile(tbl_profile, tbl_ai_meta)

        # 4. Write
        out_path = file_writer.write(tbl_profile)
        print(f"  💾 Saved      → {out_path}")

        # 5. Lineage
        lineage.record(tbl_profile, "success", CONFIG["targets"])

        elapsed = round(time.time() - table_start, 1)
        print(f"  ⏱️  {elapsed}s | Tags: {tbl_ai_meta.get('tags',[])}")

        results.append({
            "Table"       : table_name,
            "Rows"        : f"{tbl_profile.row_count:,}",
            "Cols"        : len(tbl_profile.columns),
            "Domain"      : tbl_profile.ai_domain,
            "Sensitivity" : tbl_profile.ai_sensitivity,
            "PII Cols"    : len([c for c in tbl_profile.columns if c.is_pii]),
            "Status"      : "✅ Success",
            "Time (s)"    : elapsed,
        })
        success_count += 1

    except Exception as e:
        print(f"  ❌ FAILED: {e}")
        results.append({
            "Table": table_name, "Rows": "—", "Cols": "—",
            "Domain": "—", "Sensitivity": "—", "PII Cols": "—",
            "Status": f"❌ {str(e)[:40]}", "Time (s)": "—"
        })
        failed_count += 1

total_duration = round(time.time() - pipeline_start, 1)
summary_path   = lineage.write_summary(len(all_tables), success_count, failed_count, total_duration)
print(f"\n{'='*60}")
print(f"🏁 Pipeline Complete | {success_count}/{len(all_tables)} succeeded | {total_duration}s total")

---
## 📊 Step 12: Pipeline Results Summary

In [ ]:
results_df = pd.DataFrame(results)

display(HTML(f"""
<div style='background:#0d1117;padding:15px;border-radius:8px;color:white;font-family:sans-serif'>
  <b style='font-size:18px'>🏁 MetaBridge Run Summary</b><br><br>
  📊 Tables Processed : <b style='color:#58a6ff'>{len(all_tables)}</b><br>
  ✅ Success          : <b style='color:#3fb950'>{success_count}</b><br>
  ❌ Failed           : <b style='color:#f85149'>{failed_count}</b><br>
  ⏱️ Total Duration   : <b style='color:#ffd700'>{total_duration}s</b><br>
  💾 Output Folder    : <b style='color:#a5d6ff'>{CONFIG['output_dir']}</b>
</div>
"""))

display(HTML("<h4>📋 Per-Table Results</h4>"))
display(results_df.style.set_properties(**{
    'background-color': '#161b22',
    'color': 'white',
    'border': '1px solid #30363d'
}).set_table_styles([{
    'selector': 'th',
    'props': [('background-color', '#21262d'), ('color', '#58a6ff'), ('font-weight', 'bold')]
}]))

---
## 🔗 Step 13: Lineage Log Viewer

In [ ]:
lineage_records = []
with open(lineage.log_path, "r") as f:
    for line in f:
        lineage_records.append(json.loads(line.strip()))

lineage_df = pd.DataFrame(lineage_records)[[
    "source_table", "row_count", "columns_count",
    "ai_domain", "ai_sensitivity", "pii_columns",
    "status", "timestamp", "run_id"
]]

display(HTML("<h4>🔗 Full Lineage Log</h4>"))
display(lineage_df)

---
## 🔍 Step 14: Inspect a Catalog JSON Output

In [ ]:
# Show the generated catalog JSON for Customers
inspect_table = "Customers"
json_path = f"{CONFIG['output_dir']}/{inspect_table}_catalog.json"

if os.path.exists(json_path):
    with open(json_path) as f:
        catalog_json = json.load(f)

    # Summary view
    ai = catalog_json["ai_metadata"]
    display(HTML(f"""
    <div style='background:#0d1117;padding:15px;border-radius:8px;color:white'>
      <b style='color:#58a6ff;font-size:16px'>📄 {inspect_table}_catalog.json</b><br><br>
      <b>Description  :</b> {ai['description']}<br>
      <b>Purpose      :</b> {ai['purpose']}<br>
      <b>Domain       :</b> <span style='color:#ffd700'>{ai['domain']}</span><br>
      <b>Sensitivity  :</b> <span style='color:#ff7b72'>{ai['sensitivity']}</span><br>
      <b>Tags         :</b> {', '.join(ai['tags'])}<br>
      <b>Lineage RunID:</b> <span style='color:#8b949e'>{catalog_json['lineage']['run_id']}</span><br>
      <b>Source Path  :</b> <span style='color:#a5d6ff'>{catalog_json['lineage']['source_to_catalog']}</span>
    </div>
    """))

    print("\n📦 Full JSON:")
    print(json.dumps(catalog_json, indent=2))
else:
    print(f"⚠️ {json_path} not found — run the pipeline first (Step 11)")

---
## 📁 Step 15: List All Output Files

In [ ]:
output_files = sorted(os.listdir(CONFIG["output_dir"]))
file_info = []
for fn in output_files:
    fp = os.path.join(CONFIG["output_dir"], fn)
    size = os.path.getsize(fp)
    file_info.append({"File": fn, "Size (bytes)": size, "Size (KB)": round(size/1024, 1)})

files_df = pd.DataFrame(file_info)
total_kb  = files_df["Size (KB)"].sum()

display(HTML(f"<h4>📁 Output: {CONFIG['output_dir']} | {len(output_files)} files | {total_kb:.1f} KB total</h4>"))
display(files_df)

---
## ☁️ Step 16: Push to Microsoft Purview (Optional)

Fill `CONFIG['purview_account']` and set `'targets': ['purview']` then run this cell.  
Requires: `pip install azure-identity` and Azure credentials configured.

In [ ]:
if CONFIG.get("purview_account"):
    import requests
    from azure.identity import DefaultAzureCredential

    class PurviewWriter:
        def __init__(self, account):
            self.endpoint   = f"https://{account}.purview.azure.com"
            self.credential = DefaultAzureCredential()

        def _token(self):
            return self.credential.get_token("https://purview.azure.com/.default").token

        def write(self, profile):
            qn = f"{profile.source_system}/{profile.database}/{profile.schema}/{profile.table_name}"
            entity = {
                "entity": {
                    "typeName": "rdbms_table",
                    "attributes": {
                        "qualifiedName" : qn,
                        "name"          : profile.table_name,
                        "description"   : profile.ai_table_description,
                        "userDescription": profile.ai_table_purpose,
                        "comment"       : f"Domain:{profile.ai_domain}|Sensitivity:{profile.ai_sensitivity}",
                        "columns": [{
                            "typeName": "rdbms_column",
                            "attributes": {
                                "qualifiedName": f"{qn}/{c.name}",
                                "name"         : c.name,
                                "description"  : c.ai_description,
                                "data_type"    : c.data_type,
                                "isNullable"   : c.nullable,
                            }
                        } for c in profile.columns]
                    }
                }
            }
            resp = requests.post(
                f"{self.endpoint}/catalog/api/atlas/v2/entity",
                json=entity,
                headers={"Authorization": f"Bearer {self._token()}", "Content-Type": "application/json"}
            )
            resp.raise_for_status()
            return resp.json()

    purview_writer = PurviewWriter(CONFIG["purview_account"])
    # Load profiles from saved JSONs and push
    for fn in os.listdir(CONFIG["output_dir"]):
        if fn.endswith("_catalog.json") and not fn.startswith("run_"):
            with open(f"{CONFIG['output_dir']}/{fn}") as f:
                cat = json.load(f)
            print(f"Pushing {cat['source']['table']} to Purview...")
            # (Rebuild profile from JSON for push — or store profiles in memory)
    print("✅ Purview push complete")
else:
    display(HTML("""
    <div style='background:#21262d;padding:12px;border-left:4px solid #f0883e;border-radius:4px;color:white'>
      ⏭️ <b>Purview push skipped</b> — set <code>CONFIG['purview_account']</code> to enable.<br>
      Also add <code>'purview'</code> to <code>CONFIG['targets']</code>.
    </div>
    """))

---
## 🧱 Step 17: Push to Databricks Unity Catalog (Optional)

Fill `CONFIG['databricks_host']`, `databricks_token`, `databricks_warehouse_id`.  
Requires: `pip install databricks-sdk`

In [ ]:
if CONFIG.get("databricks_host") and CONFIG.get("databricks_token"):
    from databricks.sdk import WorkspaceClient

    class UnityCatalogWriter:
        def __init__(self, host, token, catalog, warehouse_id):
            self.client       = WorkspaceClient(host=host, token=token)
            self.catalog      = catalog
            self.warehouse_id = warehouse_id

        def write(self, profile):
            full = f"{self.catalog}.{profile.schema}.{profile.table_name}"
            self.client.tables.update(full_name=full, comment=profile.ai_table_description)
            for col in profile.columns:
                if col.ai_description:
                    desc = col.ai_description.replace("'", "''")
                    self.client.statement_execution.execute_statement(
                        warehouse_id=self.warehouse_id,
                        statement=(
                            f"ALTER TABLE `{self.catalog}`.`{profile.schema}`.`{profile.table_name}` "
                            f"ALTER COLUMN `{col.name}` COMMENT '{desc}'"
                        ),
                        wait_timeout="30s"
                    )
            print(f"  ✅ Unity Catalog updated: {full}")

    uc_writer = UnityCatalogWriter(
        CONFIG["databricks_host"], CONFIG["databricks_token"],
        CONFIG["uc_catalog"], CONFIG["databricks_warehouse_id"]
    )
    print("✅ Unity Catalog writer ready — run uc_writer.write(profile) per table")
else:
    display(HTML("""
    <div style='background:#21262d;padding:12px;border-left:4px solid #58a6ff;border-radius:4px;color:white'>
      ⏭️ <b>Unity Catalog push skipped</b> — set <code>databricks_host</code> + <code>databricks_token</code> in CONFIG.
    </div>
    """))

---
## ✅ You're Done!

### What was generated:
- `metabridge_output/<table>_catalog.json` — enriched metadata per table
- `metabridge_output/lineage_log.jsonl` — full audit lineage trail
- `metabridge_output/run_summary.json` — pipeline run summary

### Next steps:
1. **Add your LLM key** in Step 3 CONFIG and re-run Step 11
2. **Add Purview credentials** and run Step 16 to push to Microsoft Purview
3. **Add Databricks credentials** and run Step 17 to push to Unity Catalog
4. **Swap the connector** — replace `NorthwindProfiler` with `SQLConnector` for Oracle/SAP/PostgreSQL sources
5. **Schedule** — wrap pipeline in ADF Custom Activity or Airflow DAG

> 💡 **Ollama tip:** Set `llm_provider: ollama` for a fully local, free, air-gapped run with Llama 3.